# CENG313 Introduction to Data Science — Assignment 1: EDA & Baseline Models
## Competition: [CAFA 6 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-6-protein-function-prediction)

This notebook carries out an Exploratory Data Analysis (EDA) and implements Baseline Models for the CAFA 6 Protein Function Prediction challenge.

All the reusable code is under the `src/` folder.

## 1. Environment Setup & Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Ensure project root is in sys.path so `src` can be imported cleanly
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    resolve_data_paths,
    parse_fasta,
    load_train_terms,
    load_taxonomy,
    load_ia_weights,
    parse_go_obo_summary,
    inspect_non_canonical_amino_acids,
    ProteinFeatureExtractor,
    build_multilabel_targets
)

from src.models import (
    NaiveFrequencyBaseline
)

from src.utils import (
    plot_aspect_and_top_terms,
    plot_long_tail_and_labels_per_protein,
    plot_sequence_length_and_aa_composition,
    plot_taxonomy_and_ia,
    compute_fmax,
    compute_ontology_fmax
)

plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

## 2. Dataset Configuration
Set `DATA_DIR` to point to the directory containing the competition files.

In [ ]:
DATA_DIR = Path("data/cafa-6-protein-function-prediction")
PATHS = resolve_data_paths(DATA_DIR)

for name, path in PATHS.items():
    print(f"{name:<18}: {path} (exists={path.exists()})")

In [ ]:
# Load core dataset files
train_seq_df = parse_fasta(PATHS["train_fasta"])
test_seq_df = parse_fasta(PATHS["test_fasta"])
terms_df = load_train_terms(PATHS["train_terms"])

tax_df = load_taxonomy(PATHS["train_taxonomy"]) if PATHS["train_taxonomy"].exists() else None
test_tax_df = load_taxonomy(PATHS["test_taxonomy"]) if PATHS["test_taxonomy"].exists() else None
ia_df = load_ia_weights(PATHS["ia"]) if PATHS["ia"].exists() else None
obo_summary = parse_go_obo_summary(PATHS["go_obo"])

print(f"Training sequences : {len(train_seq_df):,}")
print(f"Test sequences     : {len(test_seq_df):,}")
print(f"Total annotations  : {len(terms_df):,}")
print(f"Unique GO terms    : {terms_df['term'].nunique():,}")
print(f"GO OBO summary     : {obo_summary}")

## 3. Exploratory Data Analysis

### 3.1 Data Integrity and Missing/Corrupted Value Check
We verify whether there are any missing (`NaN`) records, inspect sequence length distributions, and scan all protein sequences for non-canonical/ambiguous IUPAC residues (`B`, `U`, `X`, `Z`, `O`, `J`) or corrupted/invalid characters outside the 20 standard amino acids (`ACDEFGHIKLMNPQRSTVWY`).

In [ ]:
print("---- Missing Values in train_seq_df ")
print(train_seq_df.isna().sum())
print("\n---- Missing Values in terms_df ")
print(terms_df.isna().sum())

print("\n---- Sequence Length Summary Statistics (Train vs Test) ")
summary_stats = pd.DataFrame({
    "Train_Length": train_seq_df["length"].describe(),
    "Test_Length": test_seq_df["length"].describe(),
})
display(summary_stats)

print("\n--- Non-Canonical / Invalid Amino Acid Overview ---")
aa_overview_df, aa_residue_df = inspect_non_canonical_amino_acids({
    "Train": train_seq_df,
    "Test": test_seq_df,
})
display(aa_overview_df)

print("\n--- Detailed Breakdown of Non-Canonical Amino Acid Residues ---")
display(aa_residue_df)

### 3.2 Sub-Ontology (Aspect) Breakdown & Top GO Terms
Gene Ontology terms are organized into three independent sub-ontologies:
- **BPO**: Biological Process
- **CCO**: Cellular Component
- **MFO**: Molecular Function

In [ ]:
fig1 = plot_aspect_and_top_terms(terms_df, top_n=20)
plt.show()

### 3.3 Long Tail Label Distribution and Multi Label Cardinality

A challenge in CAFA 6 is severe label imbalance: a small number of general GO terms appear in thousands of proteins, while thousands of specific GO terms appear in only a couple of proteins.

In [ ]:
fig2 = plot_long_tail_and_labels_per_protein(terms_df)
plt.show()

labels_per_protein = terms_df.groupby("EntryID")["term"].count()
print(f"Mean GO terms per protein  : {labels_per_protein.mean():.2f}")
print(f"Median GO terms per protein: {labels_per_protein.median():.1f}")
print(f"Max GO terms on a protein  : {labels_per_protein.max()}")

### 3.4 Protein Sequence Lengths and Amino Acid Composition

In [ ]:
fig3 = plot_sequence_length_and_aa_composition(train_seq_df, test_seq_df)
plt.show()

### 3.5 Organism Taxonomy and Information Accretion (IA) Weights
Information Accretion (IA) weights measure how much additional information a specific GO term provides given its parent terms in the ontology DAG. Rare/specific terms have high IA weights.

In [ ]:
fig4 = plot_taxonomy_and_ia(tax_df, ia_df, top_n_taxa=15)
plt.show()

## 4. Feature Engineering and Target Matrix Construction
1. **Features (X)**:
   - 20 canonical amino acid relative frequencies (AAC) + non-standard residue ratio + log(1 + L) sequence length
   - Character k-mer TF-IDF features (k=2 dipeptides and k=3 tripeptides)
   - Top 50 organism Taxon ID one hot indicators
2. **Targets (Y)**:
   - Binary multi label matrix covering the Top300 most frequent GO terms across BPO, CCO, and MFO.

In [ ]:
TOP_K_TERMS = 300

# Merge taxonomy IDs onto training and test sequence DataFrames
if tax_df is not None:
    train_merged = train_seq_df.merge(tax_df, on="EntryID", how="left")
    train_merged["taxon_id"] = train_merged["taxon_id"].fillna("unknown")
else:
    train_merged = train_seq_df.copy()
    train_merged["taxon_id"] = "unknown"

# Filter to proteins that have at least one annotation in train_terms.tsv
annotated_ids = set(terms_df["EntryID"].unique())
train_merged = train_merged[train_merged["EntryID"].isin(annotated_ids)].reset_index(drop=True)

Y_all, selected_terms, term_to_aspect = build_multilabel_targets(
    protein_ids=train_merged["EntryID"],
    terms_df=terms_df,
    top_k=TOP_K_TERMS,
)

ia_map = dict(zip(ia_df["term"], ia_df["ia"])) if ia_df is not None else None

# Train / Validation split (80% / 20%)
train_idx, val_idx = train_test_split(
    np.arange(len(train_merged)),
    test_size=0.20,
    random_state=42,
)

extractor = ProteinFeatureExtractor(ngram_range=(2, 3), max_kmer_features=1500, top_taxa=50)
X_train = extractor.fit_transform(
    train_merged.loc[train_idx, "sequence"],
    train_merged.loc[train_idx, "taxon_id"],
)
X_val = extractor.transform(
    train_merged.loc[val_idx, "sequence"],
    train_merged.loc[val_idx, "taxon_id"],
)
Y_train = Y_all[train_idx]
Y_val = Y_all[val_idx]

print(f"X_train shape: {X_train.shape} | Y_train shape: {Y_train.shape}")
print(f"X_val shape  : {X_val.shape} | Y_val shape  : {Y_val.shape}")

## 5. Baseline Model Training and Validation
1. Baseline 0 - Naive Frequency Prior (NaiveFrequencyBaseline): Predicts GO term frequencies without using sequence features.

In [ ]:
# Baseline 0: Naive Frequency Prior
naive_model = NaiveFrequencyBaseline().fit(X_train, Y_train)
val_prob_naive = naive_model.predict_proba(X_val)
df_eval_naive = compute_ontology_fmax(Y_val, val_prob_naive, selected_terms, term_to_aspect, ia_map)
print("=== Baseline 0: Naive Frequency Prior ===")
display(df_eval_naive)